# Sweep Mode A — Pre-build sweep

A *sweep* runs many cases in one call and collects the results. openTEPES has three sweep modes; this notebook covers the first.

**Mode A (pre-build)** reads each case from its own input folder and builds and solves it independently. Use it when your cases are genuinely different folders, such as separate scenarios with many changed inputs.

When cases share most of their data, two faster modes avoid re-reading and rebuilding:
- [Mode B](5.2-Sweep-Mode-B-InMemory.ipynb) reads one baseline once and perturbs it.
- [Mode C](5.3-Sweep-Mode-C-Resolve.ipynb) builds the model once and re-solves it.

## The question

All three sweep notebooks answer the same question: **how does the total system cost change if electricity demand is higher?**

For Mode A we build two case folders: a base case and a high-demand case (demand scaled up by 10%).

In [1]:
import os, shutil
import pandas as pd

def coarse_copy(src, dst):
    """Copy a case folder and set a coarse time resolution so the example runs fast."""
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    p = os.path.join(dst, "oT_Data_Parameter_9n.csv")
    df = pd.read_csv(p)
    df.loc[:, "TimeStep"] = 24   # coarse resolution, just to keep this tutorial quick
    df.to_csv(p, index=False)

# Base scenario
coarse_copy("9n", "sweepA_base/9n")

# High-demand scenario: same case, electricity demand scaled up by 10%
coarse_copy("9n", "sweepA_high/9n")
dem = "sweepA_high/9n/oT_Data_Demand_9n.csv"
df = pd.read_csv(dem)
node_cols = [c for c in df.columns if c.startswith("Node_")]
df[node_cols] = df[node_cols] * 1.10
df.to_csv(dem, index=False)
print("Two scenario folders are ready.")

Two scenario folders are ready.


## Run the sweep

A `Case` describes one entry of the sweep: the folder that holds it (`dir_name`), the case name (`case_name`), where to write its results (`out_path`), and a short `label` for the summary.

`openTEPES_Runner.run` builds and solves each case and returns one summary row per case, in the order you listed them.

In [2]:
from openTEPES import openTEPES_Runner, openTEPES_Cases

cases = [
    openTEPES_Cases.Case("sweepA_base", "9n", out_path="sweepA_base/out", label="base"),
    openTEPES_Cases.Case("sweepA_high", "9n", out_path="sweepA_high/out", label="high_demand"),
]

records = openTEPES_Runner.run(
    cases, "appsi_highs",
    mode="pre-build", backend="serial",
    pIndOutputResults=0, pIndLogConsole=0,
)

Input data                             ****
Reading the CSV files                  ...  0 s
Reading    input data                  ...  0 s


Setting up input data                  ...  1 s
Setting up variables                   ...  0 s
Total cost o.f.      model formulation ****
Investment elec      model formulation ****
Period 2030, Scenario sc01, Stage st1
Generation oper o.f. model formulation ****
Investment & operation var constraints ****


Inertia, oper resr, demand constraints ****
Storage   scheduling       constraints ****
Unit commitment            constraints ****
Ramp and min up/down time  constraints ****


Network    switching model constraints ****
Network    operation model constraints ****
Problem solving                        #### 1


Termination condition:  optimal
  Total system                 cost [MEUR]  159.21117655177736  Constraints 41136  Variables 50966  Seconds 3
***** Period: 2030, Scenario: sc01, Stage: st1 ******
  Total generation  investment cost [MEUR]  0
  Total generation  retirement cost [MEUR]  0
  Total reservoir   investment cost [MEUR]  0.0
  Total network     investment cost [MEUR]  4.041225328215885
  Total H2   pipe   investment cost [MEUR]  0.0
  Total heat pipe   investment cost [MEUR]  0.0
  Total generation  operation  cost [MEUR]  155.16550622161148
  Total consumption operation  cost [MEUR]  0.0028532975065870933
  Total emission               cost [MEUR]  0.0
  Total network losses penalty cost [MEUR]  0.0015917044432770533
  Total reliability electr     cost [MEUR]  0.0
Writing            investment results  ...  0 s
Writing          cost summary results  ...  0 s
Writing           KPI summary results  ...  0 s


...

Investment & operation var constraints ****
Inertia, oper resr, demand constraints ****
Storage   scheduling       constraints ****
Unit commitment            constraints ****
Ramp and min up/down time  constraints ****
Network    switching model constraints ****
Network    operation model constraints ****


Problem solving                        #### 1


Termination condition:  optimal
  Total system                 cost [MEUR]  199.22790748954316  Constraints 41136  Variables 50966  Seconds 3
***** Period: 2030, Scenario: sc01, Stage: st1 ******
  Total generation  investment cost [MEUR]  0
  Total generation  retirement cost [MEUR]  0
  Total reservoir   investment cost [MEUR]  0.0
  Total network     investment cost [MEUR]  5.3371090320017105
  Total H2   pipe   investment cost [MEUR]  0.0
  Total heat pipe   investment cost [MEUR]  0.0
  Total generation  operation  cost [MEUR]  193.8862041750987
  Total consumption operation  cost [MEUR]  0.0028444742272815595
  Total emission               cost [MEUR]  0.0
  Total network losses penalty cost [MEUR]  0.001749808215068684
  Total reliability electr     cost [MEUR]  0.0
Writing            investment results  ...  0 s
Writing          cost summary results  ...  0 s
Writing           KPI summary results  ...  0 s


Writing elect network summary results  ...  0 s


Writing              economic results  ...  0 s


In [3]:
pd.DataFrame(records)[["label", "status", "total_cost_meur"]]

,label,status,total_cost_meur
0,base,optimal,159.211177
1,high_demand,optimal,199.227907


## What just happened

Each case was solved on its own, and the total system cost rises in the high-demand scenario. A case that fails gets `status="error"` instead of stopping the whole sweep.

## Run the sweep in parallel

The cases are independent, so they can solve at the same time on several CPU cores. Switch the backend to `"multiprocessing"` and choose how many workers to use. The summaries still come back in input order, so the result matches the serial run, just faster when there are many cases.

Binder runs on Linux, where this works directly.

In [4]:
records_parallel = openTEPES_Runner.run(
    cases, "appsi_highs",
    mode="pre-build", backend="multiprocessing", n_workers=2,
    pIndOutputResults=0, pIndLogConsole=0,
)

Input data                             ****
Reading the CSV files                  ...  0 s
Reading    input data                  ...  0 s
Setting up input data                  ...  1 s
Setting up variables                   ...  0 s
Total cost o.f.      model formulation ****
Investment elec      model formulation ****
Period 2030, Scenario sc01, Stage st1
Generation oper o.f. model formulation ****
Investment & operation var constraints ****
Inertia, oper resr, demand constraints ****
Storage   scheduling       constraints ****
Unit commitment            constraints ****
Ramp and min up/down time  constraints ****
Network    switching model constraints ****
Network    operation model constraints ****
Problem solving                        #### 1
Input data                             ****
Reading the CSV files                  ...  0 s
Reading    input data                  ...  0 s
Setting up input data                  ...  1 s
Setting up variables                   ...  0 s
Tota

Running HiGHS 1.14.0 (git hash: 7df0786): Copyright (c) 2026 under MIT licence terms
Running HiGHS 1.14.0 (git hash: 7df0786): Copyright (c) 2026 under MIT licence terms


LP has 41136 rows; 57516 cols; 166697 nonzeros
Coefficient ranges:
  Matrix  [1e-04, 2e+02]
  Cost    [1e+00, 1e+00]
  Bound   [1e-03, 1e+03]
  RHS     [1e-03, 5e+02]
Presolving model
36761 rows, 45853 cols, 138280 nonzeros 0s
28396 rows, 37478 cols, 126116 nonzeros 0s
Dependent equations search running on 5389 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
26482 rows, 34839 cols, 131299 nonzeros 0s
Presolve reductions: rows 26482(-14654); columns 34839(-22677); nonzeros 131299(-35398) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.1s
      17019     1.9922790749e+02 Pr: 0(0); Du: 0(7.16127e-13) 1.6s
      17019     1.9922790749e+02 Pr: 0(0); Du: 0(7.16127e-13) 1.6s

Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Simplex   iterations: 17019
Obj

LP has 41136 rows; 57516 cols; 166697 nonzeros
Coefficient ranges:
  Matrix  [1e-04, 2e+02]
  Cost    [1e+00, 1e+00]
  Bound   [9e-04, 1e+03]
  RHS     [9e-04, 5e+02]
Presolving model
36762 rows, 45854 cols, 138283 nonzeros 0s
28394 rows, 37474 cols, 126094 nonzeros 0s
Dependent equations search running on 5388 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
26479 rows, 34834 cols, 131282 nonzeros 0s
Presolve reductions: rows 26479(-14657); columns 34834(-22682); nonzeros 131282(-35415) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.1s
      17066     1.5921117655e+02 Pr: 0(0) 2.0s
      17066     1.5921117655e+02 Pr: 0(0) 2.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Simplex   iterations: 17066
Objective value     :  1.5921117655e+02
P-D

Termination condition:  optimal
  Total system                 cost [MEUR]  159.21117655177747  Constraints 41136  Variables 50966  Seconds 3
***** Period: 2030, Scenario: sc01, Stage: st1 ******
  Total generation  investment cost [MEUR]  0
  Total generation  retirement cost [MEUR]  0
  Total reservoir   investment cost [MEUR]  0.0
  Total network     investment cost [MEUR]  4.041225328215885
  Total H2   pipe   investment cost [MEUR]  0.0
  Total heat pipe   investment cost [MEUR]  0.0
  Total generation  operation  cost [MEUR]  155.16550622161174
  Total consumption operation  cost [MEUR]  0.0028532975065870907
  Total emission               cost [MEUR]  0.0
  Total network losses penalty cost [MEUR]  0.001591704443277054
  Total reliability electr     cost [MEUR]  0.0
Writing            investment results  ...  0 s
Writing          cost summary results  ...  0 s
Writing           KPI summary results  ...  0 s
Writing elect network summary results  ...  0 s
Writing              eco

In [5]:
pd.DataFrame(records_parallel)[["label", "status", "total_cost_meur"]]

,label,status,total_cost_meur
0,base,optimal,159.211177
1,high_demand,optimal,199.227907


`backend="joblib"` works the same way (it needs `pip install joblib`). On a real sweep with many cases, raise `n_workers` to the number of cores you want to use.

## Collect every case's results into one table

So far we only compared the total cost. To compare the full results across cases, pass `aggregate_to` a folder. After the sweep, openTEPES stacks each result table across all cases into one long table, `oT_Sweep_<table>.csv`, with a leading `case` column.

This reads the per-case results from disk, so we turn result writing on with `pIndOutputResults="Yes"`.

In [6]:
records = openTEPES_Runner.run(
    cases, "appsi_highs",
    mode="pre-build", backend="serial",
    pIndOutputResults="Yes", pIndLogConsole="No",
    aggregate_to="sweepA_merged",
)

Input data                             ****
Reading the CSV files                  ...  0 s
Reading    input data                  ...  0 s


Setting up input data                  ...  1 s


Setting up variables                   ...  0 s
Total cost o.f.      model formulation ****
Investment elec      model formulation ****
Period 2030, Scenario sc01, Stage st1
Generation oper o.f. model formulation ****
Investment & operation var constraints ****
Inertia, oper resr, demand constraints ****
Storage   scheduling       constraints ****
Unit commitment            constraints ****


Ramp and min up/down time  constraints ****
Network    switching model constraints ****
Network    operation model constraints ****


Problem solving                        #### 1


...

Writing           KPI summary results  ...  0 s
Writing elect network summary results  ...  0 s
Writing           reliability indexes  ...  0 s
Writing           flexibility results  ...  0 s


/private/tmp/claude-501/-Users-philias-ai-research-repos-openTEPES-tutorial/8f6d4ac5-9ff1-45bc-8a3c-5cc562abe3f1/scratchpad/venv312/lib/python3.12/site-packages/altair/utils/core.py:264: UserWarning: I don't know how to infer vegalite type from 'empty'.  Defaulting to nominal.
  warnings.warn(


Writing  generation operation results  ...  0 s
Writing         ESS operation results  ...  0 s
Writing elect netwk operation results  ...  0 s


Writing  marginal information results  ...  0 s


Writing              economic results  ...  0 s
Plotting electricity network     maps  ...  0 s


Each `oT_Sweep_*.csv` in `sweepA_merged/` holds every case's rows for one result, told apart by the `case` column. For example, the total generation per technology in each case:

In [7]:
gen = pd.read_csv("sweepA_merged/oT_Sweep_TechnologyGeneration.csv")
tech = [c for c in gen.columns if c not in ("case", "Scenario", "Period", "LoadLevel")]
gen.groupby("case")[tech].sum().round(1)

,Coal,ESS,Gas,Nuclear,Oil,RES
case,,,,,,
base,0.0,10699.9,52416.4,280862.4,0.0,57746.6
high_demand,0.0,10666.8,91387.5,280862.4,0.0,57746.6


When your cases share most of their inputs, Mode A re-reads the same data for each one. The next notebook, [Mode B](5.2-Sweep-Mode-B-InMemory.ipynb), avoids that.